[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/fluids/kirchhoff/kirchhoff.ipynb)

# Kirchhoff's Equations

This example computes the added-mass inertia and free motion of a solid ellipsoid moving through a still, inviscid, incompressible fluid under Kirchhoff's equations. A body moving through water pushes the surrounding fluid aside, and the kinetic energy of that displaced flow acts as anisotropic extra inertia carried along with the body, which is called its added mass. Because an elongated body pushes far less fluid aside when moving end-on than when moving broadside, its total momentum tilts away from its direction of travel, producing the destabilizing Munk moment that turns an unstabilized hull broadside to its path.

The ellipsoid is modelled by its shape map on directions, holding its squared semi-axes along the body axes, so the confocal ellipsoids form the one-parameter pencil `shape + widening * Direction`. Averaged over that pencil and weighted by the ratio of body volume to confocal volume, the inverse pencil gives the fluid's response to shifts and its outermorphism on couples gives the response to turns. Combined with the solid's own inertia, both assemble into a single duality map from twists to forques ([Gunn, 2011](https://arxiv.org/pdf/1101.4542); [Dorst and De Keninck, 2022](https://bivector.net/PGAdyn.pdf)) whose pairing `rate & inertia(rate)` is twice the kinetic energy of body and fluid together, and under zero external force the body coasts at the twist `mobility(motor << impulse)` of the space-fixed impulse seen in the body frame.

In matrix Lie–Poisson and Kirchhoff–Lamb vector notation, the combined inertia of body and fluid is a split $6 \times 6$ added-mass block matrix $\begin{pmatrix} M & C^T \\ C & J \end{pmatrix}$ mapping body-frame linear and angular velocities $(\mathbf V, \boldsymbol\Omega)$ to linear and angular impulses $(\mathbf P, \boldsymbol\Pi)$, which evolve under the coupled cross-product equations $\dot{\mathbf P} = \mathbf P \times \boldsymbol\Omega$ and $\dot{\boldsymbol\Pi} = \boldsymbol\Pi \times \boldsymbol\Omega + \mathbf P \times \mathbf V$ with principal diagonal entries built axis by axis from Lamb's elliptic integrals $\alpha_0 = abc \int_0^\infty \frac{d\lambda}{(a^2 + \lambda)\sqrt{(a^2+\lambda)(b^2+\lambda)(c^2+\lambda)}}$.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from itertools import islice

import numpy as np

from numga import NumpyContext, dim, stack
from numga.algebras import PGA3D
from examples.util.animation import save_figure, show_states
from examples.fluids.kirchhoff import render, scenarios

ga = PGA3D
mv = NumpyContext(ga, units=True).multivector
Scalar = ga.gatype.scalar()                                # Scalar+
Point = ga.gatype.antivector()                             # Antivector-
# Offsets and point velocities are points at infinity: degenerate antivectors with intrinsic orientation.
Direction = Point.derive.degenerate                        # Antivector-
Twist = ga.gatype.bivector()                               # Bivector+
Forque = ga.gatype.antibivector()                          # Antibivector-
# Couples are forques along a line at infinity: the angular momentum of a pure turn.
Couple = Forque.derive.degenerate                          # Antibivector-
Motor = ga.gatype.rotor()                                  # Rotor+
# A stretch map carrying the unit sphere onto the ellipsoid.
Stretch = ga.gatype((Direction, Direction))                # Direction- <- Direction-
# A shape map holding the squared semi-axes along the body's axes.
Shape = ga.gatype((Direction, Direction))                  # Direction- <- Direction-
# An inertia duality map sending twists to forques.
Inertia = ga.gatype((Forque, Twist))                       # Forque- <- Twist+

# The body's centre, the coordinate planes through it, and the directions normal to those planes.
centre = mv.point([0, 0, 0], unit=dim.length)              # [] Point-, 1 | length
axes = stack([mv.x, mv.y, mv.z])                           # [3] Plane+
x, y, z = axes                                             # [] Plane+, Plane+, Plane+
directions = axes.dual()                                   # [3] Direction-
# The velocity of the centre under a twist, and the couple extracted from its rotational part.
centre_velocity = centre.commutator(Twist)                 # [] Direction- <- Twist+
turn_couple = directions.regressive(directions.commutator(Twist)).sum(axis=-1) / 2  # [] Couple- <- Twist+

## Confocal pencil and added-fluid inertia

Translational momentum is a line through the body's centre formed by joining the centre with the linear momentum direction, whereas rotational momentum is an ideal couple. While the solid's translational mass is isotropic and its rotational inertia on each coordinate plane is proportional to the sum of the two squared semi-axes in that plane, the surrounding fluid's rotational inertia scales with their squared difference `axis_sum(axis_sum) - 4 * areas`, so turning an ellipsoid about an axis of symmetry sets no fluid in motion.

In [ ]:
n_aspects = 41
aspects = np.geomspace(0.1, 10, n_aspects)
n_widenings = 48

In [ ]:
def ellipsoid_stretch(semi_axes: Scalar) -> Stretch:
    """Linear stretch carrying the unit sphere onto an ellipsoid with the given semi-axes along the body axes."""
    # Scale each direction component along its corresponding axis plane by that axis's semi-axis length.
    return (directions * semi_axes * (axes & Direction)).sum(axis=-1)            # [...] Direction- <- Direction-


def quadrature(n_widenings: int) -> tuple[np.ndarray, np.ndarray]:
    """Confocal widening nodes on the positive half-line and their quadrature weights via Gauss-Legendre mapping."""
    nodes, weights = np.polynomial.legendre.leggauss(n_widenings)
    unit_nodes = (nodes + 1) / 2
    return 1 / unit_nodes**2 - 1, weights / unit_nodes**3


def total_inertia(shape: Shape, density: Scalar, fluid_density: Scalar, n_widenings: int) -> Inertia:
    """Combined solid and added-fluid inertia of an ellipsoid in an ideal fluid."""
    widenings, weights = quadrature(n_widenings)
    mean_square = shape.trace() / len(axes)                                      # [...] Scalar+
    volume = 4 / 3 * np.pi * shape.det().square_root()                           # [...] Scalar+
    # Confocal ellipsoids grow every squared semi-axis equally, weighted by body volume over confocal volume.
    confocal_shapes = shape[..., None] + (mean_square[..., None] * widenings) * Direction     # [..., n_widenings] Direction- <- Direction-
    quadrature_weights = (shape.det()[..., None] / confocal_shapes.det()).square_root() * mean_square[..., None] * weights   # [..., n_widenings] Scalar+
    shift_factors = (confocal_shapes.inverse() * quadrature_weights).sum(axis=-1)             # [...] Direction- <- Direction-
    turn_factors = (confocal_shapes.inverse().outermorphism(Couple) * quadrature_weights).sum(axis=-1)   # [...] Couple- <- Couple-
    # On each coordinate plane, `areas` multiplies the two squared semi-axes and `axis_sum` adds them via the mixed
    # outermorphism term; scaling the identity by `mean_square` keeps the sum in squared length.
    areas = shape.outermorphism(Couple)                                          # [...] Couple- <- Couple-
    axis_sum = ((mean_square * Direction + shape).outermorphism(Couple) / mean_square
                - mean_square * Couple - areas / mean_square)                    # [...] Couple- <- Couple-
    discriminant = axis_sum(axis_sum) - 4 * areas                                # [...] Couple- <- Couple-
    fluid_shift = (2 * Direction - shift_factors).solve(shift_factors)           # [...] Direction- <- Direction-
    fluid_turn = (2 * Couple - axis_sum(turn_factors)).solve(discriminant(turn_factors)) / 5  # [...] Couple- <- Couple-
    shift_inertia = (volume * density) * Direction + (volume * fluid_density) * fluid_shift   # [...] Direction- <- Direction-
    turn_inertia = (volume * density / 5) * axis_sum + (volume * fluid_density) * fluid_turn  # [...] Couple- <- Couple-
    return (centre & shift_inertia(centre_velocity)) + turn_inertia(turn_couple) # [...] Forque- <- Twist+


spheroid_axes = mv.scalar(np.stack([aspects, np.ones_like(aspects), np.ones_like(aspects)], axis=-1)[..., None], unit=dim.length)  # [n_aspects, 3] Scalar+
stretches = ellipsoid_stretch(spheroid_axes)                                     # [n_aspects] Direction- <- Direction-
shapes = stretches(stretches)                                                    # [n_aspects] Direction- <- Direction-
density_unit = dim.mass / dim.length**3
unit_density = mv.scalar([1], unit=density_unit)
zero_density = mv.scalar([0], unit=density_unit)
solid_inertia = total_inertia(shapes, unit_density, zero_density, n_widenings)[:, None]   # [n_aspects, 1] Forque- <- Twist+
fluid_inertia = total_inertia(shapes, zero_density, unit_density, n_widenings)[:, None]   # [n_aspects, 1] Forque- <- Twist+
rates = stack([mv.wx * (dim.length / dim.time), mv.wy * (dim.length / dim.time), mv.zx / dim.time])  # [3] Twist+
ratios = (rates & fluid_inertia(rates)) / (rates & solid_inertia(rates))         # [n_aspects, 3] Scalar+
save_figure(render.draw_coefficients(aspects, ratios), "kirchhoff_added_mass");

The curves show the fluid kinetic energy divided by the solid kinetic energy at equal density, with blue for shifting along the symmetry axis, orange for shifting across it, and green for turning an axis across it. At the vertical grey line, a sphere carries half the displaced fluid mass in every direction and zero rotational added inertia, whereas a flat disc moving face-on carries several times its own volume of fluid and a slender rod moving end-on carries almost none.

## Turning broadside under a fixed impulse

An ellipsoid given an impulsive blow and left to coast keeps the combined impulse of body and fluid fixed along a spatial line. At each Lie midpoint step, the body advances by the twist whose momentum equals the space-fixed impulse pulled back into the body frame. Without fluid inertia the body glides straight along the blow, whereas in the fluid a slanted ellipsoid drifts transversely off the impulse line because longitudinal motion takes less momentum than broadside motion; once the impulse line passes beside the centre, its moment swings an end-on launch past broadside and rocks a near-broadside launch around broadside. Rendering the animation takes about five seconds.

In [ ]:
body_semi_axes = mv.scalar([[3.0], [1.0], [1.0]], unit=dim.length)
density = mv.scalar(1.0, unit=density_unit)
fluid_densities = mv.scalar([[0.0], [1.0], [1.0]], unit=density_unit)
n_cases = fluid_densities.shape[0]
blow = mv.scalar(12.0, unit=dim.mass * dim.length / dim.time)
headings = np.array([0.1, 0.1, np.pi / 2 - 0.1])
dt = mv.scalar(0.16, unit=dim.time)
n_steps = 250
n_steps_per_frame = 2
n_frames = n_steps // n_steps_per_frame + 1
n_outline = 48
frame_duration_ms = 40

In [ ]:
def trajectory(motor: Motor, impulse: Forque, inertia: Inertia, dt: Scalar, n_steps: int):
    """Yield the body's motor placement from the start and after each Lie midpoint step under a space-fixed impulse."""
    mobility = inertia.inverse()                                                 # [n_cases] Twist+ <- Forque-
    yield motor
    for _ in range(n_steps):
        midpoint = motor * (mobility(motor << impulse) * (-dt / 4)).exp()        # [n_cases] Motor+
        motor = motor * (mobility(midpoint << impulse) * (-dt / 2)).exp()        # [n_cases] Motor+
        yield motor


body_stretch = ellipsoid_stretch(body_semi_axes)                                 # [] Direction- <- Direction-
inertias = total_inertia(body_stretch(body_stretch), density, fluid_densities, n_widenings)   # [n_cases] Forque- <- Twist+
long_axis = directions[0]                                                        # [] Direction-
impulse = blow * (centre & long_axis)                                            # [] Forque-
initial_motors = (mv.xy * (-headings / 2)).exp()                                 # [n_cases] Motor+
outline_angles = np.linspace(0, 2 * np.pi, n_outline, endpoint=False)
circle = (mv.xy * (-outline_angles / 2)).exp() >> long_axis                      # [n_outline] Direction-
body_outline = centre + body_stretch(circle)                                     # [n_outline] Point-
nose = centre + body_stretch(long_axis)                                          # [] Point-
gliding_states = islice(trajectory(initial_motors, impulse, inertias, dt, n_steps), 0, None, n_steps_per_frame)
motor_frames = show_states(gliding_states, render.animate_gliding, impulse, body_outline, nose,
                           name="kirchhoff_gliding", duration_ms=frame_duration_ms, n_frames=n_frames);

From top to bottom, the three lanes show the ellipsoid coasting in vacuum, launched nearly end-on in the fluid, and launched nearly broadside in the fluid, seen from above. In each lane the dashed line marks the space-fixed impulse, the solid curve traces the centre's path, and the dot marks the nose at one end of the long axis.

In [ ]:
# checks
# At aspect ratio 1 (the midpoint of the log-spaced sweep), the sphere has translational ratio 1/2 and rotational ratio 0.
sphere_index = n_aspects // 2
np.testing.assert_allclose(ratios[sphere_index].to_array(), [0.5, 0.5, 0.0], atol=1e-12)
# Total kinetic energy of body and fluid is conserved across the recorded frames within midpoint-integrator accuracy.
motors = stack(motor_frames)                                                     # [n_frames, n_cases] Motor+
body_impulses = motors << impulse                                                # [n_frames, n_cases] Forque-
energies = (body_impulses & inertias.inverse()(body_impulses)).to_array()
np.testing.assert_allclose(energies / energies[0], 1, rtol=1e-2)
# In vacuum the heading stays constant; launched nearly broadside it rocks about broadside; nearly end-on swings past broadside.
long_axes = motors >> long_axis                                                  # [n_frames, n_cases] Direction-
headings_over_time = np.arctan2((y & long_axes).to_array(), (x & long_axes).to_array())
np.testing.assert_allclose(headings_over_time[:, 0], headings[0], atol=1e-12)
assert np.abs(headings_over_time[:, 2] - np.pi / 2).max() < 2 * (np.pi / 2 - headings[2])
assert headings_over_time[:, 1].max() > np.pi / 2